# Download models from CivitAI

This notebook looks up each model on CivitAI and saves it into the matching ComfyUI folder (LoRAs in `models/loras`, checkpoints in `models/checkpoints`, ...), which is on the network volume. Downloads use aria2c with 16 connections and resume if interrupted. Refresh the model list in ComfyUI (press `R`) to see new files.

Set `CAI_API_KEY` to your CivitAI API key in the RunPod template (create one at https://civitai.com/user/account), or paste it in the next cell.

In [ ]:
import os

if not os.environ.get("CAI_API_KEY"):
    os.environ["CAI_API_KEY"] = ""  # paste your CivitAI API key between the quotes

print("CAI_API_KEY is set" if os.environ["CAI_API_KEY"] else "CAI_API_KEY is not set, some models can't be downloaded")

In [ ]:
import os, re, shutil, subprocess, requests

MODELS = "/ComfyUI/models"
# CivitAI model type -> ComfyUI models folder
FOLDERS = {"Checkpoint": "checkpoints", "LORA": "loras", "LoCon": "loras", "DoRA": "loras",
           "TextualInversion": "embeddings", "VAE": "vae", "Controlnet": "controlnet",
           "Upscaler": "upscale_models", "Hypernetwork": "hypernetworks", "MotionModule": "animatediff_models"}
API = "https://civitai.com/api/v1"
KEY = os.environ.get("CAI_API_KEY", "")
AUTH = {"Authorization": f"Bearer {KEY}"} if KEY else {}

def version_id(url):
    """Model version ID from a model page, a version link, a download link or a plain ID"""
    m = re.search(r"modelVersionId=(\d+)", url) or re.search(r"/api/download/models/(\d+)", url)
    if m:
        return m.group(1)
    m = re.search(r"/models/(\d+)", url)
    model_id = m.group(1) if m else url.strip()
    r = requests.get(f"{API}/models/{model_id}", headers=AUTH, timeout=30)
    r.raise_for_status()
    return str(r.json()["modelVersions"][0]["id"])  # newest version

def download(url, connections=16):
    r = requests.get(f"{API}/model-versions/{version_id(url)}", headers=AUTH, timeout=30)
    r.raise_for_status()
    info = r.json()
    file = next((f for f in info["files"] if f.get("primary")), info["files"][0])
    folder = os.path.join(MODELS, FOLDERS.get(info["model"]["type"], "other"))
    path = os.path.join(folder, file["name"])
    if os.path.exists(path):
        print(f"Already there: {path}")
        return
    os.makedirs(folder, exist_ok=True)
    # The download link redirects to a signed storage URL, which aria2c can fetch over many connections
    r = requests.get(file["downloadUrl"], headers=AUTH, allow_redirects=False, timeout=30)
    if r.status_code in (401, 403):
        raise RuntimeError(f"{file['name']}: CivitAI refused the download, set CAI_API_KEY")
    real_url = r.headers.get("Location", file["downloadUrl"])
    print(f"Downloading {file['name']} ({file['sizeKB'] / 1024:.0f} MB) to {folder}")
    if shutil.which("aria2c"):
        subprocess.run(["aria2c", "-x", str(connections), "-s", str(connections), "-k", "1M", "--continue=true",
                        "--console-log-level=warn", "--summary-interval=10", "-d", folder,
                        "-o", file["name"] + ".part", real_url], check=True)
    else:
        with requests.get(real_url, stream=True, timeout=60) as resp, open(path + ".part", "wb") as out:
            resp.raise_for_status()
            shutil.copyfileobj(resp.raw, out, 1 << 20)
    os.replace(path + ".part", path)
    print(f"Done: {path}")


Put the models you want in `URLS`: model pages (newest version), version links (`?modelVersionId=`), download links or model IDs all work. Files that are already there are skipped.

In [ ]:
URLS = [
    "https://civitai.com/models/4201?modelVersionId=130072",
]

for url in URLS:
    download(url)

For more options (CSV batch lists, other URL formats, choosing the target folder), install the full [civitai-bulk-downloader](https://github.com/deepratna-awale/civitai-bulk-downloader) in this pod:

```
!pip install git+https://github.com/deepratna-awale/civitai-bulk-downloader.git
```